# Przegląd lotów

Notebook czyta bazę `data/flights.db` (tworzoną przez `python src/parse.py`) i pokazuje:

1. tabelę wszystkich lotów,
2. porównanie lotów (czas, dystans, zużycie baterii),
3. przebieg jednego lotu w czasie (wysokość, prędkość, bateria, GPS),
4. mapę trasy.

> **Uwaga:** wyniki (tabele, mapa) zawierają Twoje współrzędne GPS. Przed commitem
> wyczyść wyniki: *Clear All Outputs* w VS Code.

In [ ]:
import sqlite3
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import pandas as pd

# The notebook may be started from the project root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "flights.db"
con = sqlite3.connect(DB_PATH)

# Chart style: one accent hue for data, recessive grey axes and grid,
# text in neutral ink (never in the series color)
BLUE = "#2a78d6"      # data series
INK = "#0b0b0b"       # primary text
INK2 = "#52514e"      # secondary text
MUTED = "#898781"     # axis labels / ticks
GRID = "#e1e0d9"      # gridlines
AXIS = "#c3c2b7"      # baseline / axis
SURFACE = "#fcfcfb"   # chart background

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS,
    "axes.labelcolor": INK2,
    "axes.titlecolor": INK,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "lines.linewidth": 2,
    "font.size": 10,
})

## 1. Wszystkie loty

Jeden wiersz na lot z tabeli `flights`. Dodatkowo liczymy kolumny pomocnicze:
czas lokalny startu (z nazwy pliku), minuty, kilometry i zużycie baterii w punktach procentowych.

In [ ]:
flights = pd.read_sql("SELECT * FROM flights ORDER BY flight_id", con)

# Local start time comes from the file name, e.g. "2026-08-25_[19-12-02]"
flights["start_local"] = pd.to_datetime(flights["flight_id"], format="%Y-%m-%d_[%H-%M-%S]")
flights["label"] = flights["start_local"].dt.strftime("%d.%m %H:%M")
flights["minutes"] = flights["duration_s"] / 60
flights["km"] = flights["distance_m"] / 1000
flights["battery_used_pct"] = flights["battery_start_pct"] - flights["battery_end_pct"]

summary = flights[[
    "label", "minutes", "km", "max_height_m", "max_hspeed_ms",
    "battery_start_pct", "battery_end_pct", "battery_used_pct",
    "battery_max_temp_c", "gps_p05_sats_air", "photos", "video_s",
]].rename(columns={
    "label": "start", "minutes": "czas [min]", "km": "dystans [km]",
    "max_height_m": "maks. wys. [m]", "max_hspeed_ms": "maks. pręd. [m/s]",
    "battery_start_pct": "bat. start [%]", "battery_end_pct": "bat. koniec [%]",
    "battery_used_pct": "zużycie [pp]", "battery_max_temp_c": "maks. temp. [°C]",
    "gps_p05_sats_air": "GPS p5 [sat.]",
    "photos": "zdjęcia", "video_s": "wideo [s]",
})
print(f"Lotów: {len(flights)}, łącznie {flights.minutes.sum():.0f} min i {flights.km.sum():.1f} km")
summary.round(1)

## 2. Porównanie lotów

Trzy osobne wykresy zamiast jednego z kilkoma osiami: każda miara ma swoją skalę.
Loty ułożone chronologicznie od góry.

In [ ]:
metrics = [
    ("minutes", "Czas lotu", "min"),
    ("km", "Dystans", "km"),
    ("battery_used_pct", "Zużycie baterii", "pp"),
]
# Chronological from the top: reverse order for barh (it draws bottom-up)
data = flights.iloc[::-1]

fig, axes = plt.subplots(1, len(metrics), figsize=(13, 5), sharey=True)
for ax, (col, title, unit) in zip(axes, metrics):
    ax.barh(data["label"], data[col], color=BLUE, height=0.6)
    ax.set_title(f"{title} [{unit}]")
    ax.grid(axis="y", visible=False)          # horizontal bars: vertical grid only
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(0, data[col].max() * 1.15)    # room for value labels
    for y, v in enumerate(data[col]):         # value at the bar end, in muted ink
        ax.text(v, y, f" {v:.0f}" if unit == "pp" else f" {v:.1f}", va="center", fontsize=8, color=INK2)
fig.tight_layout()
plt.show()

## 3. Jeden lot w szczegółach

Ustaw `FLIGHT_ID` na dowolny lot z tabeli powyżej (domyślnie: najdłuższy).
Czas na osi X to `OSD.flyTime` – sekundy od startu logu.

> Sygnału RC tu nie pokazujemy: dla Lito X1 biblioteka odczytuje go błędnie
> (wartości do 125 zamiast 0–100, skoki wokół stałego 90), więc wykres byłby mylący.

In [ ]:
FLIGHT_ID = flights.loc[flights["duration_s"].idxmax(), "flight_id"]
print("Lot:", FLIGHT_ID)

frames = pd.read_sql(
    """
    SELECT "OSD.flyTime" AS t, "OSD.height" AS height, "OSD.hSpeed" AS hspeed,
           "BATTERY.chargeLevel" AS battery, "OSD.gpsNum" AS gps,
           "OSD.latitude" AS lat, "OSD.longitude" AS lon, "OSD.isOnGround" AS on_ground
    FROM frames WHERE flight_id = ?
    """,
    con, params=(FLIGHT_ID,),
)
frames["minute"] = frames["t"] / 60
# Battery reports 0 before its first reading – hide those samples
frames.loc[frames["battery"] <= 0, "battery"] = None
frames.describe().round(1)

In [ ]:
panels = [
    ("height", "Wysokość nad punktem startu [m]"),
    ("hspeed", "Prędkość pozioma [m/s]"),
    ("battery", "Bateria [%]"),
    ("gps", "Liczba satelitów GPS"),
]
# Shared X (time), but every measure has its own panel and its own Y scale
fig, axes = plt.subplots(len(panels), 1, figsize=(12, 9), sharex=True)
for ax, (col, title) in zip(axes, panels):
    ax.plot(frames["minute"], frames[col], color=BLUE)
    ax.set_title(title)
    ax.grid(axis="x", visible=False)

axes[2].set_ylim(0, 100)
axes[-1].set_xlabel("Czas od startu logu [min]")
fig.tight_layout()
plt.show()

## 4. Mapa trasy

Linia trasy z punktów GPS, zielony znacznik = start, czerwony = koniec.
Mapa zapisuje się też jako HTML w `data/maps/` (folder ignorowany przez git).

In [ ]:
track = frames[(frames["lat"] != 0) & (frames["lon"] != 0)].dropna(subset=["lat", "lon"])
points = track[["lat", "lon"]].values.tolist()

m = folium.Map(tiles="OpenStreetMap")
folium.PolyLine(points, color=BLUE, weight=3).add_to(m)
folium.Marker(points[0], tooltip="Start", icon=folium.Icon(color="green")).add_to(m)
folium.Marker(points[-1], tooltip="Koniec", icon=folium.Icon(color="red")).add_to(m)
m.fit_bounds([track[["lat", "lon"]].min().tolist(), track[["lat", "lon"]].max().tolist()])

maps_dir = ROOT / "data" / "maps"
maps_dir.mkdir(exist_ok=True)
m.save(maps_dir / f"{FLIGHT_ID}.html")
m